# HIP-HOPS-LLM on Colab

**Hierarchical imprecise reliability assessment and failure propagation for
LLM-based agentic systems.**

This notebook clones the repository, installs the package, and runs the whole
pipeline: a LangGraph architecture in, fault trees and a calibrated Bayesian
network out. It needs no GPU, no API key and no data of your own — everything
runs against bundled examples.

Runtime: about two minutes, most of it the install.

Repository: https://github.com/koo-ec/HIP_HOPS_LLM

---
## 1. Install

In [ ]:
!git clone --depth=1 https://github.com/koo-ec/HIP_HOPS_LLM.git 2>&1 | tail -2
%pip install -q -e "HIP_HOPS_LLM[bayes,graph]" 2>&1 | tail -3
print("installed")

In [ ]:
import importlib, platform, sys

print("python  ", sys.version.split()[0], "on", platform.system())
for name in ("numpy", "pandas", "matplotlib", "scipy", "pyagrum", "langgraph"):
    try:
        module = importlib.import_module(name)
        print(f"{name:<12} {getattr(module, '__version__', 'present')}")
    except ImportError:
        print(f"{name:<12} not installed (optional here)")

import HIP_HOPS_LLM as H

print(f"\nHIP-HOPS-LLM {H.__version__} — {len(H.__all__)} public names")
print("Graphviz available:", H.graphviz_available())

That last line matters for the pictures further down. Graphviz is a native
binary, not a Python package, and whether a Colab image has it varies — so the
package tests for it (`dot -V`) rather than assuming. With it, pyAgrum renders
the Bayesian networks; without it, they are drawn with matplotlib and the caption
says so. Either way `bn.show()` produces a picture. `!apt-get -qq install
graphviz` forces the pyAgrum path.

---
## 2. The whole pipeline, in ten lines

A LangGraph architecture, observed benchmark outcomes, an operational profile —
and a calibrated Bayesian network at the end.

In [ ]:
from HIP_HOPS_LLM import AgenticReliabilityStudy, load_example, load_outcomes

study = AgenticReliabilityStudy(
    load_example("parallel_aggregator"),
    name="parallel agents + aggregator",
)
study.observe(load_outcomes(), profile={"short": 0.30, "medium": 0.50, "long": 0.20})
study.run()

print(study.summary())

Two things in that output are the point of the whole package.

**The common-cause group was not declared anywhere.** It was read out of the node
functions' source: both agents and the judge instantiate the same model snapshot.

**`CCF-LLM-…` is an order-1 cut set for the critical hazard.** Two agents
answering in parallel with a judge between them *looks* redundant, and most of
its cut sets really are order 2 — but a fault they share defeats the vote on its
own. An architecture diagram cannot show you that.

---
## 3. What the architecture looks like

In [ ]:
import pandas as pd

pd.DataFrame(study.system.architecture_table()).set_index("component")

In [ ]:
import matplotlib.pyplot as plt

study.plot_architecture()
plt.show()

---
## 4. The fault tree

In [ ]:
ax = study.plot("H2")
ax.figure.set_size_inches(14, 9)
plt.show()

In [ ]:
print("minimal cut sets for H2 — incorrect answer delivered and accepted:\n")
for cut in sorted(study.cut_sets("H2"), key=lambda c: (len(c), c)):
    print(f"  order {len(cut)}:  " + " + ".join(cut))

---
## 5. What was measured, and what it licenses

In [ ]:
study.calibration.evidence_frame()

In [ ]:
print("per-stratum failure rates — this is why a pooled accuracy is not enough:\n")
for name, evidence in study.evidence.items():
    rates = ",  ".join(f"{k}={v:.3f}" for k, v in evidence.by_stratum.items())
    print(f"  {name:<14} {rates}")

In [ ]:
study.calibration.to_frame()

`n = 160`, not 240: the outcome table carries a `split` column, and only the
calibration rows were used. Fitting basic-event probabilities on the evaluation
set would make every downstream number optimistic and untestable.

And the estimate is an **interval**. 160 items do not identify a point, and
HIP-LLM's hierarchical imprecise-Bayesian posterior says so rather than
pretending otherwise.

---
## 6. The Bayesian network

In [ ]:
network = study.bayesnet("H2")
print(network.summary())

print("\ntwo engines that share no code, cross-checked:")
for key, value in network.cross_check().items():
    print(f"  {key:<22} {value}")

In [ ]:
print("exact inference vs the minimal cut upper bound:\n")
for key, value in network.compare_with_cutsets(study.report.analysis("H2")).items():
    print(f"  {key:<28} {value:.6f}")
print("\nThe bound is loose because those cut sets share basic events.")
print("Together they bracket the answer.")

In [ ]:
network.show()

---
## 7. Diagnosis: why did it fail?

In [ ]:
posterior = network.posteriors({"BE-aggregator-OWN": "Fail"})
pd.Series(posterior, name="P(cause | the judge hallucinated)").head(8)

In [ ]:
network.view(evidence={"BE-aggregator-OWN": "Fail"}).show()

---
## 8. The answer, as an interval

In [ ]:
imprecise = study.imprecise_bayesnet("H2")
print(imprecise.summary())

print("\nP(H2) =", study.hazard_probability("H2"))

That is P(an incorrect answer is delivered and accepted as correct) for one
request drawn from the stated operational profile — computed by exact inference
at both ends of every basic event's interval.

Change the profile and it changes, which is exactly the point: a failure
probability quoted without the workload it is conditional on is not a claim about
the system at all.

In [ ]:
for name, profile in {
    "as measured": {"short": 0.30, "medium": 0.50, "long": 0.20},
    "harder":      {"short": 0.05, "medium": 0.15, "long": 0.80},
    "easier":      {"short": 0.70, "medium": 0.25, "long": 0.05},
}.items():
    other = AgenticReliabilityStudy(load_example("parallel_aggregator"))
    other.observe(load_outcomes(), profile=profile).run()
    print(f"  {name:<12} P(H2) in {other.hazard_probability('H2')}")

---
## 9. Does diversifying the models help?

In [ ]:
cases = {
    "as built (shared snapshot)": {},
    "cot_agent diversified": {
        "cot_agent": {"llm": "gpt-4o-mini", "runtime": "api"},
    },
    "agents and judge diversified": {
        "cot_agent": {"llm": "gpt-4o-mini", "runtime": "api"},
        "aggregator": {"llm": "claude-sonnet-4-5", "runtime": "api2"},
    },
}
for name, overrides in cases.items():
    s = AgenticReliabilityStudy(
        load_example("parallel_aggregator"), name=name, resource_overrides=overrides
    )
    s.analyse()
    analysis = s.report.analysis("H2")
    order_1 = sorted(next(iter(c)) for c in analysis.cuts.sets if len(c) == 1)
    print(f"{name:<30} P(H2)={analysis.quant.top_probability:.4f}")
    print(f"{'':<30} order-1: {order_1}\n")

**Diversifying one agent changes nothing.** The common-cause group shrinks from
three members to two, but the judge is still one of them and the judge is on the
value path alone — so `CCF-LLM-…` remains an order-1 cut set. You have to
diversify the judge too, and then `P(H2)` drops by about a fifth.

---
## 10. The FMEA, and everything saved

In [ ]:
study.fmea().head(10)

In [ ]:
for path in study.save("artifacts"):
    print(path)

---
## 11. Your own LangGraph

In [ ]:
# In your own notebook, replace the bundled example with your compiled graph and
# pass globals() so the extractor can interrogate the live model objects — which
# is what makes shared-snapshot (common-cause) detection reliable.
#
# study = AgenticReliabilityStudy(
#     graph,
#     name="my workflow",
#     globals_ns=globals(),
#     node_functions={"planner::router": route_fn},   # add_conditional_edges callables
# )
# study.analyse()                    # structure — needs no data at all
# print(study.summary())
#
# study.observe(my_outcomes, profile=my_profile).run()
# study.hazard_probability("H2")

from HIP_HOPS_LLM import describe_examples

print(describe_examples())

---
## Where next

* **Docs** — https://github.com/koo-ec/HIP_HOPS_LLM/tree/main/docs/source
* **Tutorial 7** covers pointing this at a real LangGraph application, including
  what to log and how much of it you need.
* **HIP-LLM's own API** is re-exported here, so
  `from HIP_HOPS_LLM import OperationalFailureProb` works.

If you use this, please cite both the HIP-LLM paper
([10.1016/j.ress.2026.112615](https://doi.org/10.1016/j.ress.2026.112615)) and
this repository — `CITATION.cff` has both.